# Official SwinJSCC — DIV2K → Kodak | Kaggle 2×T4 DataParallel

This notebook is deliberately built around the **official `semcomm/SwinJSCC` implementation** rather than a reimplementation of the architecture.

Official repository:
`https://github.com/semcomm/SwinJSCC`

Official high-resolution experiment:

- Training: DIV2K / HR images
- Testing: Kodak
- Input crop: 256×256
- Downsampling factor: 4
- Base model:
  - encoder depths `[2, 2, 6, 2]`
  - decoder depths `[2, 6, 2, 2]`
  - embedding dimensions `[128, 192, 256, 320]`
  - heads `[4, 6, 8, 10]`
  - window size `8`
- Channel: AWGN
- Adaptive model: `SwinJSCC_w/_SAandRA`
- Rates: `C = 32, 64, 96, 128, 192`
- SNRs: `1, 4, 7, 10, 13 dB`

The official implementation recommends first training `SwinJSCC_w/o_SAandRA` and then using it as the initialization for the complete SA+RA model.

This notebook additionally wraps the official encoder/decoder/channel so that the model's `forward()` returns **tensors only**. That is necessary for PyTorch `DataParallel`; the original official `network.py` returns Python scalar values such as CBR and SNR along with tensors, which is not a safe output structure for `DataParallel`.

Important: this notebook uses your requested DIV2K and Kodak paths. It does not mix Kodak into training.


## Dataset configuration

The requested paths are:

```python
KAGGLE_KODAK_DIR = Path(
    "/kaggle/input/datasets/sherylmehta/kodak-dataset"
)

DIV2K_DIR = Path(
    "/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr"
)
```

Training uses the DIV2K directory if it exists.

Kodak is kept exclusively for held-out evaluation.

Because the Kaggle DIV2K ISR notebook can contain generated/output folders and may contain images at several resolutions, the notebook does **not** blindly use every image. It:

1. Recursively discovers image files.
2. Reports the directory distribution and image resolutions.
3. Keeps only training images whose dimensions are at least 256×256.
4. Randomly crops those HR images to 256×256, following the spirit of the official HR loader.
5. Uses Kodak separately for evaluation, center-cropping each image to dimensions compatible with the Swin hierarchy.

The exact contents of the attached Kaggle dataset are printed by the notebook, so you can verify what is actually being trained on.


In [ ]:
import os
import sys
import math
import time
import random
import logging
import subprocess
import importlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPUs   :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# ============================================================
# EXACT DATASET PATHS REQUESTED
# ============================================================

KAGGLE_KODAK_DIR = Path(
    "/kaggle/input/datasets/sherylmehta/kodak-dataset"
)

DIV2K_DIR = Path(
    "/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr"
)

IMAGE_SIZE = 256

if not DIV2K_DIR.exists():
    raise FileNotFoundError(
        f"DID NOT FIND DIV2K DATASET:\n{DIV2K_DIR}"
    )

if not KAGGLE_KODAK_DIR.exists():
    raise FileNotFoundError(
        f"DID NOT FIND KODAK DATASET:\n{KAGGLE_KODAK_DIR}"
    )

print("DIV2K path :", DIV2K_DIR)
print("Kodak path :", KAGGLE_KODAK_DIR)


In [ ]:
# ============================================================
# DISCOVER AND AUDIT DATASET CONTENTS
# ============================================================

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp",
    ".webp", ".tif", ".tiff"
}

def discover_images(root):
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

div2k_all = discover_images(DIV2K_DIR)
kodak_all = discover_images(KAGGLE_KODAK_DIR)

print("All DIV2K-path images :", len(div2k_all))
print("All Kodak images      :", len(kodak_all))

def audit_images(files, name, limit=None):
    records = []

    check_files = files if limit is None else files[:limit]

    for p in check_files:
        try:
            with Image.open(p) as im:
                records.append({
                    "path": str(p),
                    "width": im.width,
                    "height": im.height,
                    "format": im.format
                })
        except Exception as e:
            print("Could not open:", p, "|", e)

    df = pd.DataFrame(records)

    print(f"\n{name} resolution summary")
    print("=" * 70)

    if len(df) == 0:
        print("No readable images.")
        return df

    print(df[["width", "height"]].describe())

    print("\nTop parent directories:")
    print(
        df["path"]
        .map(lambda x: str(Path(x).parent))
        .value_counts()
        .head(20)
    )

    return df

div2k_audit = audit_images(
    div2k_all,
    "DIV2K",
)

kodak_audit = audit_images(
    kodak_all,
    "Kodak",
)


In [ ]:
# ============================================================
# SELECT HR TRAINING IMAGES
# ============================================================

# The SwinJSCC official HR loader takes random 256x256 crops.
# We therefore require the source image itself to be at least
# 256x256. This removes obvious low-resolution ISR products.

def is_hr_candidate(path, minimum=IMAGE_SIZE):
    try:
        with Image.open(path) as im:
            return (
                im.width >= minimum and
                im.height >= minimum
            )
    except Exception:
        return False

train_files = [
    p for p in div2k_all
    if is_hr_candidate(p)
]

if not train_files:
    raise RuntimeError(
        "No DIV2K images >= 256x256 were found."
    )

print("DIV2K images available :", len(div2k_all))
print("HR images selected     :", len(train_files))

# Print a few examples so the actual dataset is visible.
print("\nFirst 20 selected training files:")
for p in train_files[:20]:
    print(" ", p)

# Do NOT create a Kodak training split.
# Kodak remains the held-out evaluation set.
test_files = kodak_all

print("\nKodak evaluation images:", len(test_files))


## Why the dataset handling differs slightly from the original repository

The official `HR_image` loader uses a list of HR directories and applies a random 256×256 crop followed by `ToTensor`.

The official Kodak `Datasets` loader center-crops each image to dimensions divisible by 128 and evaluates it as a whole image.

We reproduce those important behaviors here while adapting the filesystem discovery to Kaggle's mounted dataset structure.


In [ ]:
# ============================================================
# DATASETS
# ============================================================

class DIV2KRandomCrop(Dataset):
    def __init__(self, files, size=256):
        self.files = list(files)
        self.size = size

        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]

        with Image.open(path) as im:
            im = im.convert("RGB")

            if im.width < self.size or im.height < self.size:
                raise RuntimeError(
                    f"Selected HR image is smaller than {self.size}: "
                    f"{path} -> {im.size}"
                )

            # Same essential behavior as official HR_image:
            # random 256x256 crop.
            x = random.randint(
                0,
                im.width - self.size
            )

            y = random.randint(
                0,
                im.height - self.size
            )

            crop = im.crop((
                x,
                y,
                x + self.size,
                y + self.size
            ))

            # Mild augmentation is reasonable for training.
            if random.random() < 0.5:
                crop = crop.transpose(
                    Image.Transpose.FLIP_LEFT_RIGHT
                )

            if random.random() < 0.5:
                crop = crop.transpose(
                    Image.Transpose.FLIP_TOP_BOTTOM
                )

            return self.to_tensor(crop)


class KodakWholeImage(Dataset):
    def __init__(self, files):
        self.files = list(files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]

        with Image.open(path) as im:
            im = im.convert("RGB")

            # Official Datasets.py makes the dimensions divisible
            # by 128 before CenterCrop.
            w, h = im.size

            w = w - (w % 128)
            h = h - (h % 128)

            if w < 128 or h < 128:
                raise RuntimeError(
                    f"Kodak image too small: {path} -> {im.size}"
                )

            left = (im.width - w) // 2
            top = (im.height - h) // 2

            im = im.crop((
                left,
                top,
                left + w,
                top + h
            ))

            return transforms.ToTensor()(im), path.name


train_dataset = DIV2KRandomCrop(
    train_files,
    IMAGE_SIZE
)

test_dataset = KodakWholeImage(
    test_files
)

# T4 x2: 8 total samples = 4 per GPU.
# Increase to 16 only if your particular Kaggle T4 memory permits it.
BATCH_SIZE = 8

NUM_WORKERS = min(
    4,
    os.cpu_count() or 1
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

print("Train images:", len(train_dataset))
print("Test images :", len(test_dataset))
print("Train batch :", BATCH_SIZE)
print("Workers     :", NUM_WORKERS)


# Install / load the official SwinJSCC implementation

The notebook clones the official repository into `/kaggle/working`.

The important source files are:

- `net/encoder.py`
- `net/decoder.py`
- `net/channel.py`
- `net/network.py`
- `main.py`
- `data/datasets.py`

The official repository states that it was developed under Python 3.8 / PyTorch 1.9 and warns that SA+RA inference can become inconsistent with PyTorch versions newer than 1.12. We record the current Kaggle environment rather than silently claiming exact paper reproduction.


In [ ]:
# ============================================================
# OFFICIAL REPOSITORY
# ============================================================

REPO_DIR = Path(
    "/kaggle/working/SwinJSCC_official"
)

if not REPO_DIR.exists():
    result = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/semcomm/SwinJSCC.git",
            str(REPO_DIR)
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            "Could not clone official SwinJSCC repository."
        )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(REPO_DIR)
    )

from net.encoder import create_encoder
from net.decoder import create_decoder
from net.channel import Channel

print("Loaded official SwinJSCC source from:")
print(REPO_DIR)

# Verify that the expected files exist.
for rel in [
    "net/encoder.py",
    "net/decoder.py",
    "net/channel.py",
    "net/network.py",
    "main.py",
]:
    p = REPO_DIR / rel
    print(rel, "->", p.exists())


In [ ]:
# ============================================================
# EXACT OFFICIAL HR BASE CONFIGURATION
# ============================================================

MODEL_BASE = "SwinJSCC_w/o_SAandRA"
MODEL_FULL = "SwinJSCC_w/_SAandRA"

EMBED_DIMS = [128, 192, 256, 320]
DEPTHS_BASE = [2, 2, 6, 2]
DEPTHS_DECODER = [2, 6, 2, 2]
NUM_HEADS = [4, 6, 8, 10]

PATCH_SIZE = 2
WINDOW_SIZE = 8
MLP_RATIO = 4.0
DOWNSAMPLE = 4

# Official examples.
TRAIN_SNRS = [1, 4, 7, 10, 13]
TRAIN_RATES = [32, 64, 96, 128, 192]

PRETRAIN_SNR = 10
PRETRAIN_RATE = 96

# Training lengths are intentionally configurable.
# Increase for a serious experiment.
PRETRAIN_EPOCHS = 10
SA_RA_EPOCHS = 10

LR_PRETRAIN = 1e-4
LR_SARA = 1e-4

print("Encoder depths :", DEPTHS_BASE)
print("Decoder depths :", DEPTHS_DECODER)
print("Embedding dims :", EMBED_DIMS)
print("SNRs           :", TRAIN_SNRS)
print("Rates          :", TRAIN_RATES)


## Official model-size correction

This is the **Base** model, not the Small model.

For the official HR Base model:

```text
Encoder depths = [2, 2, 6, 2]
Decoder depths = [2, 6, 2, 2]
```

The official source defines the SA+RA model with `C=None` at construction because the Rate ModNet chooses the active channel count dynamically. The non-adaptive baseline uses a fixed `C`.

For a 256×256 input, the four downsampling stages produce a final spatial grid:

\[
256/16 = 16
\]

so the final Swin feature contains:

\[
16\times16\times320=81,920
\]

continuous feature values per image before rate masking.


In [ ]:
# ============================================================
# OFFICIAL ENCODER / DECODER ARGUMENTS
# ============================================================

def make_encoder_kwargs(model_name, C):
    return dict(
        model=model_name,
        img_size=(IMAGE_SIZE, IMAGE_SIZE),
        patch_size=PATCH_SIZE,
        in_chans=3,
        embed_dims=EMBED_DIMS,
        depths=DEPTHS_BASE,
        num_heads=NUM_HEADS,
        C=C,
        window_size=WINDOW_SIZE,
        mlp_ratio=MLP_RATIO,
        qkv_bias=True,
        qk_scale=None,
        norm_layer=nn.LayerNorm,
        patch_norm=True,
    )


def make_decoder_kwargs(model_name, C):
    return dict(
        model=model_name,
        img_size=(IMAGE_SIZE, IMAGE_SIZE),
        embed_dims=[320, 256, 192, 128],
        depths=DEPTHS_DECODER,
        num_heads=[10, 8, 6, 4],
        C=C,
        window_size=WINDOW_SIZE,
        mlp_ratio=MLP_RATIO,
        qkv_bias=True,
        qk_scale=None,
        norm_layer=nn.LayerNorm,
        patch_norm=True,
    )


enc_base = make_encoder_kwargs(
    MODEL_BASE,
    PRETRAIN_RATE
)

dec_base = make_decoder_kwargs(
    MODEL_BASE,
    PRETRAIN_RATE
)

enc_full = make_encoder_kwargs(
    MODEL_FULL,
    None
)

dec_full = make_decoder_kwargs(
    MODEL_FULL,
    None
)

print("Baseline encoder C:", enc_base["C"])
print("SA+RA encoder C  :", enc_full["C"])


# DataParallel-safe official SwinJSCC wrapper

The official `net/network.py` returns:

```text
reconstruction,
CBR,
SNR,
MSE,
loss
```

The last four values include Python/nonscalar objects that are inconvenient for `nn.DataParallel`.

This wrapper deliberately keeps the **official encoder, decoder and channel implementation** but returns:

```text
reconstruction
feature
mask
```

all as tensors.

This is not changing the SwinJSCC mathematics. It is changing only the Python interface around the official modules so that two GPU replicas can be gathered safely.

For AWGN, the official `Channel.gaussian_noise_layer()` already creates noise using `input_layer.get_device()`, so the random channel noise follows the DataParallel replica's GPU.


In [ ]:
# ============================================================
# OFFICIAL CONFIG OBJECT
# ============================================================

class SwinConfig:
    pass


class SwinArgs:
    pass


def make_channel_args(
    model_name,
    rate_string,
    snr_string,
):
    args = SwinArgs()

    args.model = model_name
    args.C = rate_string
    args.multiple_snr = snr_string
    args.channel_type = "awgn"

    return args


def make_channel_config():
    config = SwinConfig()

    # These fields are actually consumed by the official code.
    config.pass_channel = True
    config.CUDA = torch.cuda.is_available()
    config.device = torch.device(
        "cuda:0"
        if torch.cuda.is_available()
        else "cpu"
    )
    config.norm = False

    # The official Channel constructor checks config.logger.
    config.logger = None

    config.downsample = DOWNSAMPLE

    return config


In [ ]:
# ============================================================
# OFFICIAL SWINJSCC WRAPPER
# ============================================================

class OfficialSwinJSCCDataParallel(nn.Module):

    def __init__(
        self,
        model_name,
        encoder_kwargs,
        decoder_kwargs,
        rate_string,
        snr_string,
    ):
        super().__init__()

        self.model_name = model_name

        # Official modules.
        self.encoder = create_encoder(
            **encoder_kwargs
        )

        self.decoder = create_decoder(
            **decoder_kwargs
        )

        self.channel_config = make_channel_config()

        self.channel_args = make_channel_args(
            model_name,
            rate_string,
            snr_string,
        )

        # Official Channel.
        self.channel = Channel(
            self.channel_args,
            self.channel_config
        )

        self.H = 0
        self.W = 0

    def _update_resolution_if_needed(self, H, W):

        if H != self.H or W != self.W:

            self.encoder.update_resolution(
                H,
                W
            )

            self.decoder.update_resolution(
                H // (2 ** DOWNSAMPLE),
                W // (2 ** DOWNSAMPLE)
            )

            self.H = H
            self.W = W

    def forward(
        self,
        input_image,
        given_SNR,
        given_rate
    ):
        B, _, H, W = input_image.shape

        self._update_resolution_if_needed(
            H,
            W
        )

        chan_param = given_SNR
        channel_number = given_rate

        if self.model_name in [
            "SwinJSCC_w/o_SAandRA",
            "SwinJSCC_w/_SA",
        ]:

            feature = self.encoder(
                input_image,
                chan_param,
                channel_number,
                self.model_name
            )

            noisy_feature = self.channel.forward(
                feature,
                chan_param
            )

            mask = torch.ones_like(
                feature
            )

        elif self.model_name in [
            "SwinJSCC_w/_RA",
            "SwinJSCC_w/_SAandRA",
        ]:

            feature, mask = self.encoder(
                input_image,
                chan_param,
                channel_number,
                self.model_name
            )

            avg_pwr = (
                torch.sum(feature ** 2)
                /
                mask.sum().clamp_min(1e-12)
            )

            noisy_feature = self.channel.forward(
                feature,
                chan_param,
                avg_pwr
            )

            noisy_feature = (
                noisy_feature * mask
            )

        else:
            raise ValueError(
                f"Unsupported model: {self.model_name}"
            )

        reconstruction = self.decoder(
            noisy_feature,
            chan_param,
            self.model_name
        )

        reconstruction = reconstruction.clamp(
            0.0,
            1.0
        )

        # DataParallel-safe:
        # every returned object is a tensor.
        return (
            reconstruction,
            feature,
            mask
        )


In [ ]:
# ============================================================
# DATA PARALLEL
# ============================================================

def make_parallel(model):
    model = model.cuda(0)

    if torch.cuda.device_count() >= 2:
        print(
            "Using DataParallel on:",
            list(range(torch.cuda.device_count()))
        )

        return nn.DataParallel(
            model,
            device_ids=list(
                range(torch.cuda.device_count())
            ),
            output_device=0
        )

    print("Only one CUDA device detected.")
    return model


baseline_model = make_parallel(
    OfficialSwinJSCCDataParallel(
        MODEL_BASE,
        enc_base,
        dec_base,
        rate_string=str(PRETRAIN_RATE),
        snr_string=str(PRETRAIN_SNR)
    )
)

sara_model = make_parallel(
    OfficialSwinJSCCDataParallel(
        MODEL_FULL,
        enc_full,
        dec_full,
        rate_string=",".join(
            map(str, TRAIN_RATES)
        ),
        snr_string=",".join(
            map(str, TRAIN_SNRS)
        )
    )
)

print(
    "Baseline DataParallel:",
    isinstance(
        baseline_model,
        nn.DataParallel
    )
)

print(
    "SA+RA DataParallel:",
    isinstance(
        sara_model,
        nn.DataParallel
    )
)


In [ ]:
# ============================================================
# MODEL PARAMETER COUNT
# ============================================================

def parameter_count(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )

print(
    "Baseline parameters:",
    f"{parameter_count(baseline_model)/1e6:.2f} M"
)

print(
    "SA+RA parameters:",
    f"{parameter_count(sara_model)/1e6:.2f} M"
)


In [ ]:
# ============================================================
# 2×T4 FORWARD SANITY CHECK
# ============================================================

sample = next(
    iter(train_loader)
)[:BATCH_SIZE].cuda(
    0,
    non_blocking=True
)

baseline_model.eval()
sara_model.eval()

with torch.no_grad():

    base_recon, base_feature, base_mask = (
        baseline_model(
            sample,
            PRETRAIN_SNR,
            PRETRAIN_RATE
        )
    )

    sara_recon, sara_feature, sara_mask = (
        sara_model(
            sample,
            10,
            96
        )
    )

print("Input       :", tuple(sample.shape))
print()
print("BASELINE")
print("recon       :", tuple(base_recon.shape))
print("feature     :", tuple(base_feature.shape))
print("mask        :", tuple(base_mask.shape))
print()
print("SA + RA")
print("recon       :", tuple(sara_recon.shape))
print("feature     :", tuple(sara_feature.shape))
print("mask        :", tuple(sara_mask.shape))

assert base_recon.shape == sample.shape
assert sara_recon.shape == sample.shape
assert sara_feature.ndim == 3
assert sara_mask.shape == sara_feature.shape
assert sara_feature.shape[-1] == 320

print()
print("Forward sanity check PASSED.")


In [ ]:
# ============================================================
# VERIFY BOTH GPU REPLICAS RECEIVE WORK
# ============================================================

print("CUDA device count:", torch.cuda.device_count())

if isinstance(sara_model, nn.DataParallel):
    print(
        "DataParallel device_ids:",
        sara_model.device_ids
    )
    print(
        "DataParallel output_device:",
        sara_model.output_device
    )

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)} | "
        f"allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(i)/1024**3:.2f} GB"
    )

print("\nFor live utilization during training:")
print("Run: !nvidia-smi")


# Stage 1 — official fixed-rate baseline pretraining

For the high-resolution configuration, the official repository recommends first training:

```text
SwinJSCC_w/o_SAandRA
```

at a fixed SNR and fixed C, then using that model to initialize the full SA+RA model.

We use the official example:

```text
SNR = 10 dB
C   = 96
AWGN
```


In [ ]:
# ============================================================
# STAGE 1 TRAINING
# ============================================================

optimizer_base = torch.optim.Adam(
    baseline_model.parameters(),
    lr=LR_PRETRAIN
)

use_amp = torch.cuda.is_available()

scaler_base = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

pretrain_history = []

for epoch in range(PRETRAIN_EPOCHS):

    baseline_model.train()

    running_loss = 0.0
    samples_seen = 0

    start = time.time()

    for images in train_loader:

        images = images.cuda(
            0,
            non_blocking=True
        )

        optimizer_base.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=use_amp
        ):

            reconstruction, _, _ = (
                baseline_model(
                    images,
                    PRETRAIN_SNR,
                    PRETRAIN_RATE
                )
            )

            loss = F.mse_loss(
                reconstruction,
                images
            )

        scaler_base.scale(
            loss
        ).backward()

        scaler_base.unscale_(
            optimizer_base
        )

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(),
            1.0
        )

        scaler_base.step(
            optimizer_base
        )

        scaler_base.update()

        running_loss += (
            loss.detach().item()
            * images.size(0)
        )

        samples_seen += images.size(0)

    epoch_loss = (
        running_loss
        /
        max(samples_seen, 1)
    )

    pretrain_history.append(
        epoch_loss
    )

    print(
        f"[Baseline] "
        f"Epoch {epoch+1}/{PRETRAIN_EPOCHS} | "
        f"MSE={epoch_loss:.6f} | "
        f"time={time.time()-start:.1f}s"
    )


In [ ]:
# ============================================================
# STAGE 1 LOSS PLOT
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(pretrain_history) + 1),
    pretrain_history,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("SwinJSCC Baseline Pretraining Loss")
plt.grid(True)
plt.show()


# Stage 2 — full SwinJSCC with Channel ModNet + Rate ModNet

The full model is:

```text
SwinJSCC_w/_SAandRA
```

It is trained across:

```text
SNR:  1, 4, 7, 10, 13 dB
C:    32, 64, 96, 128, 192
```

The Rate ModNet chooses the active feature channels according to the requested rate, while Channel ModNet adapts the latent representation to the channel condition.


In [ ]:
# ============================================================
# TRANSFER STAGE-1 WEIGHTS
# ============================================================

base_unwrapped = (
    baseline_model.module
    if isinstance(
        baseline_model,
        nn.DataParallel
    )
    else baseline_model
)

sara_unwrapped = (
    sara_model.module
    if isinstance(
        sara_model,
        nn.DataParallel
    )
    else sara_model
)

base_state = base_unwrapped.state_dict()
sara_state = sara_unwrapped.state_dict()

compatible = {
    k: v
    for k, v in base_state.items()
    if (
        k in sara_state
        and sara_state[k].shape == v.shape
    )
}

sara_state.update(
    compatible
)

sara_unwrapped.load_state_dict(
    sara_state,
    strict=False
)

print(
    "Transferred compatible tensors:",
    len(compatible)
)


In [ ]:
# ============================================================
# STAGE 2 TRAINING
# ============================================================

optimizer_sara = torch.optim.Adam(
    sara_model.parameters(),
    lr=LR_SARA
)

scaler_sara = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

sara_history = []

for epoch in range(SA_RA_EPOCHS):

    sara_model.train()

    running_loss = 0.0
    samples_seen = 0

    start = time.time()

    for images in train_loader:

        images = images.cuda(
            0,
            non_blocking=True
        )

        snr = random.choice(
            TRAIN_SNRS
        )

        rate = random.choice(
            TRAIN_RATES
        )

        optimizer_sara.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=use_amp
        ):

            reconstruction, _, _ = (
                sara_model(
                    images,
                    snr,
                    rate
                )
            )

            loss = F.mse_loss(
                reconstruction,
                images
            )

        scaler_sara.scale(
            loss
        ).backward()

        scaler_sara.unscale_(
            optimizer_sara
        )

        torch.nn.utils.clip_grad_norm_(
            sara_model.parameters(),
            1.0
        )

        scaler_sara.step(
            optimizer_sara
        )

        scaler_sara.update()

        running_loss += (
            loss.detach().item()
            * images.size(0)
        )

        samples_seen += images.size(0)

    epoch_loss = (
        running_loss
        /
        max(samples_seen, 1)
    )

    sara_history.append(
        epoch_loss
    )

    print(
        f"[SA+RA] "
        f"Epoch {epoch+1}/{SA_RA_EPOCHS} | "
        f"MSE={epoch_loss:.6f} | "
        f"time={time.time()-start:.1f}s"
    )


In [ ]:
# ============================================================
# STAGE 2 LOSS PLOT
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(sara_history) + 1),
    sara_history,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("SwinJSCC SA+RA Training Loss")
plt.grid(True)
plt.show()


# Evaluation

We evaluate the trained SA+RA model on the **held-out Kodak dataset** for every combination of:

\[
SNR\in\{1,4,7,10,13\}
\]

and

\[
C\in\{32,64,96,128,192\}.
\]

The official SwinJSCC CBR for HR images is:

\[
CBR=
\frac{C}
{2\cdot3\cdot2^{2\cdot4}}
=
\frac{C}{1536}.
\]

This is the communication-rate metric used by the original implementation.

We will separately report the memory size of the continuous FP32 latent. That is **not** the same thing as the communication bitrate.


In [ ]:
# ============================================================
# PSNR / EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_model(
    model,
    loader,
    snr,
    rate
):

    model.eval()

    total_squared_error = 0.0
    total_elements = 0

    for images, names in loader:

        images = images.cuda(
            0,
            non_blocking=True
        )

        reconstruction, _, _ = (
            model(
                images,
                snr,
                rate
            )
        )

        # Pixel-domain MSE on [0,1].
        total_squared_error += (
            F.mse_loss(
                reconstruction,
                images,
                reduction="sum"
            ).item()
        )

        total_elements += images.numel()

    mse = (
        total_squared_error
        /
        max(total_elements, 1)
    )

    psnr_value = (
        10.0
        * math.log10(
            1.0 / max(mse, 1e-12)
        )
    )

    return mse, psnr_value


results = []

for snr in TRAIN_SNRS:

    for rate in TRAIN_RATES:

        mse_value, psnr_value = (
            evaluate_model(
                sara_model,
                test_loader,
                snr,
                rate
            )
        )

        cbr = (
            rate
            /
            (
                2
                * 3
                * 2 ** (DOWNSAMPLE * 2)
            )
        )

        results.append({
            "SNR_dB": snr,
            "C": rate,
            "CBR": cbr,
            "MSE": mse_value,
            "PSNR_dB": psnr_value,
        })

        print(
            f"SNR={snr:>2} dB | "
            f"C={rate:>3} | "
            f"CBR={cbr:.6f} | "
            f"PSNR={psnr_value:.3f} dB"
        )

results_df = pd.DataFrame(
    results
)

display(results_df)


In [ ]:
# ============================================================
# PSNR HEATMAP
# ============================================================

heatmap = results_df.pivot(
    index="SNR_dB",
    columns="C",
    values="PSNR_dB"
)

plt.figure(figsize=(9, 5))

plt.imshow(
    heatmap.values,
    aspect="auto"
)

plt.xticks(
    range(len(heatmap.columns)),
    heatmap.columns
)

plt.yticks(
    range(len(heatmap.index)),
    heatmap.index
)

plt.xlabel("C / bottleneck dimension")
plt.ylabel("SNR (dB)")
plt.title("SwinJSCC SA+RA — Kodak PSNR")

plt.colorbar(
    label="PSNR (dB)"
)

plt.show()


In [ ]:
# ============================================================
# PSNR vs CBR
# ============================================================

plt.figure(figsize=(9, 6))

for snr in TRAIN_SNRS:

    subset = (
        results_df[
            results_df["SNR_dB"] == snr
        ]
        .sort_values("CBR")
    )

    plt.plot(
        subset["CBR"],
        subset["PSNR_dB"],
        marker="o",
        label=f"SNR={snr} dB"
    )

plt.xlabel("CBR")
plt.ylabel("PSNR (dB)")
plt.title("SwinJSCC Rate–Distortion Performance")
plt.grid(True)
plt.legend()
plt.show()


# Representation and compression report

This section answers the exact question:

> How much compression does SwinJSCC actually perform?

We report four different quantities.

### 1. Original RGB image

For a 256×256 RGB image:

\[
256\times256\times3\times8
=
1,572,864\text{ bits}
=
192\text{ KiB}.
\]

### 2. Continuous FP32 latent

For the Base HR architecture:

\[
16\times16\times320
=
81,920
\]

FP32 values.

Therefore:

\[
81,920\times32
=
2,621,440\text{ bits}
=
320\text{ KiB}.
\]

This is **tensor memory**, not the transmitted JSCC bitrate.

### 3. Rate-selected latent

For a selected \(C\), the Rate ModNet activates:

\[
16\times16\times C
\]

channel symbols.

### 4. Official SwinJSCC CBR

For HR:

\[
CBR=\frac{C}{1536}.
\]

This is the correct communication-rate measure for comparing SwinJSCC configurations.


In [ ]:
# ============================================================
# COMPRESSION / LATENT REPORT
# ============================================================

@torch.no_grad()
def inspect_latent(
    model,
    image,
    snr,
    rate
):
    model.eval()

    recon, feature, mask = model(
        image.cuda(0),
        snr,
        rate
    )

    return (
        recon,
        feature,
        mask
    )


example_image, example_name = (
    test_dataset[0]
)

example_batch = (
    example_image
    .unsqueeze(0)
)

recon, feature, mask = (
    inspect_latent(
        sara_model,
        example_batch,
        10,
        96
    )
)

B, N, latent_dim = feature.shape

selected_active = (
    mask.sum(dim=-1)
)

active_per_token = (
    selected_active[0]
    .detach()
    .cpu()
    .numpy()
)

active_channels_observed = int(
    round(
        float(
            np.median(
                active_per_token
            )
        )
    )
)

original_bits = (
    IMAGE_SIZE
    * IMAGE_SIZE
    * 3
    * 8
)

continuous_fp32_bits = (
    N
    * latent_dim
    * 32
)

active_fp32_bits = (
    N
    * active_channels_observed
    * 32
)

official_cbr = (
    active_channels_observed
    /
    (
        2
        * 3
        * 2 ** (DOWNSAMPLE * 2)
    )
)

print("=" * 90)
print("SWINJSCC REPRESENTATION / COMPRESSION REPORT")
print("=" * 90)

print(
    "Example image:               ",
    example_name
)

print(
    "Input crop used for training:",
    f"{IMAGE_SIZE} × {IMAGE_SIZE}"
)

print(
    "Original RGB payload:        ",
    f"{original_bits:,} bits "
    f"({original_bits/8/1024:.2f} KiB)"
)

print(
    "Continuous latent shape:     ",
    tuple(feature.shape)
)

print(
    "Latent values/image:         ",
    f"{N * latent_dim:,}"
)

print(
    "Continuous FP32 latent:      ",
    f"{continuous_fp32_bits:,} bits "
    f"({continuous_fp32_bits/8/1024:.2f} KiB)"
)

print(
    "Requested C:                 ",
    96
)

print(
    "Observed active channels:    ",
    active_channels_observed
)

print(
    "Active FP32 latent:          ",
    f"{active_fp32_bits:,} bits "
    f"({active_fp32_bits/8/1024:.2f} KiB)"
)

print(
    "Official SwinJSCC CBR:       ",
    f"{official_cbr:.6f}"
)

print(
    "CBR percentage:              ",
    f"{official_cbr*100:.3f}%"
)

print(
    "Original / continuous FP32:  ",
    f"{original_bits/continuous_fp32_bits:.4f}x"
)

print(
    "Original / active FP32:      ",
    f"{original_bits/active_fp32_bits:.4f}x"
)

print("=" * 90)

print(
    "\nIMPORTANT:"
)

print(
    "The FP32 latent can be larger than the raw image in memory."
)

print(
    "That does NOT mean SwinJSCC has negative compression."
)

print(
    "SwinJSCC transmits normalized real-valued channel symbols, "
    "and its rate is represented by CBR, not by storing the "
    "PyTorch FP32 tensor as a conventional file."
)


In [ ]:
# ============================================================
# OFFICIAL CBR TABLE
# ============================================================

compression_table = []

original_bits_256 = (
    256 * 256 * 3 * 8
)

for C in TRAIN_RATES:

    channel_symbols = (
        16 * 16 * C
    )

    cbr = (
        C
        /
        (
            2
            * 3
            * 2 ** (DOWNSAMPLE * 2)
        )
    )

    active_fp32_bits = (
        channel_symbols
        * 32
    )

    compression_table.append({
        "C": C,
        "CBR": cbr,
        "CBR_percent": cbr * 100,
        "active_channel_symbols": channel_symbols,
        "active_FP32_KiB": (
            active_fp32_bits
            / 8
            / 1024
        ),
        "official_rate_vs_raw_bpp": (
            cbr * 24
        ),
    })

compression_df = pd.DataFrame(
    compression_table
)

display(compression_df)


## Why the CBR table is not a conventional "file compression ratio"

SwinJSCC is a **joint source-channel coding** system.

The encoder produces continuous real-valued symbols. The channel model then normalizes those symbols and maps pairs of real values into complex channel symbols.

The official implementation defines:

\[
CBR=\frac{C}{2\cdot3\cdot2^{2i}},
\]

with \(i=4\) for high-resolution images.

Therefore, for example:

\[
C=96
\Rightarrow
CBR=0.0625.
\]

It is incorrect to say that the model literally creates a `192 KiB → 12 KiB` compressed file. That would be a different digital compression system.

For our later **codebook experiment**, this distinction becomes crucial: once we replace continuous channel symbols with discrete codebook indices, we can calculate an actual digital index payload in bits. That will be a new representation layered on top of SwinJSCC, not the original SwinJSCC bitrate.


# Reconstruction examples


In [ ]:
# ============================================================
# ORIGINAL VS RECONSTRUCTION
# ============================================================

@torch.no_grad()
def get_reconstruction(
    model,
    image,
    snr,
    rate
):
    model.eval()

    recon, _, _ = model(
        image.unsqueeze(0).cuda(0),
        snr,
        rate
    )

    return recon[0].detach().cpu()


visual_count = min(
    4,
    len(test_dataset)
)

fig, axes = plt.subplots(
    2,
    visual_count,
    figsize=(4 * visual_count, 7)
)

if visual_count == 1:
    axes = np.array(axes).reshape(2, 1)

for i in range(visual_count):

    original, name = (
        test_dataset[i]
    )

    reconstructed = (
        get_reconstruction(
            sara_model,
            original,
            10,
            96
        )
    )

    axes[0, i].imshow(
        original.permute(1, 2, 0)
    )

    axes[0, i].set_title(
        f"Original\n{name}"
    )

    axes[0, i].axis("off")

    axes[1, i].imshow(
        reconstructed.permute(1, 2, 0)
    )

    axes[1, i].set_title(
        "SwinJSCC\nSNR=10 dB, C=96"
    )

    axes[1, i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# SAVE CHECKPOINTS + RESULTS
# ============================================================

EXPORT_DIR = Path(
    "/kaggle/working/swinjscc_final_results"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

def unwrap(model):
    return (
        model.module
        if isinstance(
            model,
            nn.DataParallel
        )
        else model
    )

torch.save(
    {
        "state_dict": unwrap(
            baseline_model
        ).state_dict(),
        "model": MODEL_BASE,
        "image_size": IMAGE_SIZE,
        "dataset": str(DIV2K_DIR),
        "snr": PRETRAIN_SNR,
        "C": PRETRAIN_RATE,
    },
    EXPORT_DIR
    / "SwinJSCC_w_o_SAandRA_DIV2K.pt"
)

torch.save(
    {
        "state_dict": unwrap(
            sara_model
        ).state_dict(),
        "model": MODEL_FULL,
        "image_size": IMAGE_SIZE,
        "dataset": str(DIV2K_DIR),
        "testset": str(KAGGLE_KODAK_DIR),
        "snrs": TRAIN_SNRS,
        "rates": TRAIN_RATES,
    },
    EXPORT_DIR
    / "SwinJSCC_w_SAandRA_DIV2K_Kodak.pt"
)

results_df.to_csv(
    EXPORT_DIR
    / "kodak_evaluation.csv",
    index=False
)

compression_df.to_csv(
    EXPORT_DIR
    / "compression_table.csv",
    index=False
)

print(
    "Results saved to:",
    EXPORT_DIR
)


# Final experimental statement

After this notebook finishes, the experiment is:

\[
\boxed{
DIV2K\ HR
\rightarrow
256\times256\ crop
\rightarrow
SwinJSCC\ Encoder
\rightarrow
Channel/Rate\ ModNet
\rightarrow
AWGN
\rightarrow
SwinJSCC\ Decoder
\rightarrow
Kodak\ evaluation
}
\]

The resulting model is the baseline we will later modify for the semantic-codebook experiment.

Do **not** add the codebook to this notebook yet.

First establish that this baseline trains correctly and produces sensible:

- reconstruction images,
- PSNR vs SNR,
- PSNR vs CBR,
- latent shapes,
- and rate measurements.

Only then should we insert vector quantization/codebook processing between the SwinJSCC latent and the channel.
